# Arctic Times — Subscriber Churn Prediction

Train a churn model on subscriber behavior, evaluate performance, and deploy as a Python UDTF.

**Stack:** Snowpark Python + scikit-learn + Snowflake Model Registry

**Key message for demo:** Data never leaves Snowflake. Training runs inside the warehouse. The model is callable as a SQL function.

In [1]:
# Connect to Snowflake via Snowpark
from snowflake.snowpark import Session
import snowflake.snowpark.functions as F

session = Session.builder.config("connection_name", "lemondetrial").create()
session.use_database("ARCTIC_TIMES")
session.use_schema("RAW")
print(f"Connected: {session.get_current_account()} / {session.get_current_database()}")

Connected: "ANHTRTW-AE78140" / "ARCTIC_TIMES"


## 1. Feature Engineering

Build training features from subscriber behavior — all computed inside Snowflake.

In [2]:
# Load subscriber data as Snowpark DataFrame
subscribers = session.table("SUBSCRIBERS")

# Feature engineering — all runs inside Snowflake, no data moves
features_df = subscribers.select(
    F.col("USER_ID"),
    F.datediff("day", F.col("LAST_LOGIN"), F.current_date()).alias("DAYS_SINCE_LOGIN"),
    F.col("ARTICLES_READ_30D"),
    F.datediff("month", F.col("START_DATE"), F.current_date()).alias("TENURE_MONTHS"),
    F.col("AVG_SESSION_SEC"),
    F.col("PAYWALL_BOUNCES_30D"),
    F.col("SUBSCRIPTION_TYPE"),
    F.col("SEGMENT"),
    F.col("CHURN_FLAG").cast("int").alias("CHURNED")  # target
)

print(f"Training dataset: {features_df.count()} subscribers")
features_df.show(5)

Training dataset: 80000 subscribers


---------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"USER_ID"   |"DAYS_SINCE_LOGIN"  |"ARTICLES_READ_30D"  |"TENURE_MONTHS"  |"AVG_SESSION_SEC"  |"PAYWALL_BOUNCES_30D"  |"SUBSCRIPTION_TYPE"  |"SEGMENT"  |"CHURNED"  |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------
|USR_040001  |53                  |75                   |55               |89                 |3                      |digital_only         |regular    |1          |
|USR_040002  |28                  |119                  |34               |848                |6                      |digital_only         |regular    |0          |
|USR_040003  |58                  |83                   |51               |468                |25                     |standard             |regular    |1          |
|USR

In [3]:
# Pull to pandas for sklearn training
pdf = features_df.to_pandas()

# Encode categoricals
pdf["SUB_TYPE_ENC"] = pdf["SUBSCRIPTION_TYPE"].map({
    "trial": 0, "weekend": 1, "essential": 2,
    "premium_monthly": 3, "premium_annual": 4
}).fillna(2)

# Define features and target
feature_cols = [
    "DAYS_SINCE_LOGIN", "ARTICLES_READ_30D", "TENURE_MONTHS",
    "AVG_SESSION_SEC", "PAYWALL_BOUNCES_30D", "SUB_TYPE_ENC"
]
X = pdf[feature_cols].fillna(0)
y = pdf["CHURNED"]

print(f"Features shape: {X.shape}")
print(f"Churn rate: {y.mean():.1%}")

Features shape: (80000, 6)
Churn rate: 43.7%


## 2. Model Training

GradientBoosting classifier — handles class imbalance well for subscription churn.

In [4]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train model
model = GradientBoostingClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.1,
    subsample=0.8, random_state=42
)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=== Model Performance ===")
print(f"ROC AUC: {roc_auc_score(y_test, y_proba):.3f}")
print()
print(classification_report(y_test, y_pred, target_names=["Active", "Churned"]))

=== Model Performance ===
ROC AUC: 0.723

              precision    recall  f1-score   support

      Active       0.69      0.72      0.71      9000
     Churned       0.62      0.59      0.61      7000

    accuracy                           0.66     16000
   macro avg       0.66      0.66      0.66     16000
weighted avg       0.66      0.66      0.66     16000



In [5]:
# Feature importance — what drives churn?
import pandas as pd

importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("=== Feature Importance ===")
print(importance.to_string(index=False))
print()
print("Key insight: Days since last login is the #1 churn predictor.")
print("Actionable: Re-engagement campaign for users inactive > 14 days.")

=== Feature Importance ===
            feature  importance
   DAYS_SINCE_LOGIN    0.654768
  ARTICLES_READ_30D    0.170774
PAYWALL_BOUNCES_30D    0.079463
    AVG_SESSION_SEC    0.063494
      TENURE_MONTHS    0.031500
       SUB_TYPE_ENC    0.000000

Key insight: Days since last login is the #1 churn predictor.
Actionable: Re-engagement campaign for users inactive > 14 days.


## 3. Deploy as Python UDTF

Serialize the trained model and deploy it as a SQL-callable function. Any analyst can use it without Python.

In [6]:
import joblib, os

# Serialize model
model_path = "/tmp/churn_model.pkl"
joblib.dump(model, model_path)
print(f"Model serialized: {os.path.getsize(model_path) / 1024:.1f} KB")

# Upload to Snowflake stage
session.file.put(model_path, "@ARCTIC_TIMES.ML.MODELS", auto_compress=False, overwrite=True)
print("Uploaded to @ARCTIC_TIMES.ML.MODELS/churn_model.pkl")

Model serialized: 471.0 KB


Uploaded to @ARCTIC_TIMES.ML.MODELS/churn_model.pkl


In [7]:
# Deploy as UDTF — callable from SQL
session.sql("""
CREATE OR REPLACE FUNCTION ARCTIC_TIMES.ML.PREDICT_CHURN(
    days_since_last_login INT, articles_read_30d INT,
    subscription_months INT, avg_session_sec FLOAT, paywall_bounces_30d INT
)
RETURNS TABLE (churn_probability FLOAT, risk_segment VARCHAR)
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('scikit-learn==1.9.0', 'pandas', 'numpy', 'joblib', 'cachetools')
IMPORTS = ('@ARCTIC_TIMES.ML.MODELS/churn_model.pkl')
HANDLER = 'ChurnPredictor'
AS $$
import numpy as np, joblib, sys, os

class ChurnPredictor:
    def __init__(self):
        import_dir = sys._xoptions.get("snowflake_import_directory", "/tmp")
        self.model = joblib.load(os.path.join(import_dir, "churn_model.pkl"))

    def process(self, days_since_last_login, articles_read_30d,
                subscription_months, avg_session_sec, paywall_bounces_30d):
        features = np.array([[
            days_since_last_login or 0, articles_read_30d or 0,
            subscription_months or 0, avg_session_sec or 0.0,
            paywall_bounces_30d or 0, 2  # default sub_type
        ]])
        prob = float(self.model.predict_proba(features)[0][1])
        segment = 'HIGH' if prob > 0.7 else 'MEDIUM' if prob > 0.4 else 'LOW'
        yield (round(prob, 4), segment)
$$
""").collect()
print("UDTF deployed: ARCTIC_TIMES.ML.PREDICT_CHURN")

UDTF deployed: ARCTIC_TIMES.ML.PREDICT_CHURN


In [8]:
# Test: find high-value subscribers at churn risk
results = session.sql("""
    SELECT s.user_id, s.subscription_type, s.ltv_estimated_eur,
           c.churn_probability, c.risk_segment
    FROM ARCTIC_TIMES.RAW.SUBSCRIBERS s,
         TABLE(ARCTIC_TIMES.ML.PREDICT_CHURN(
             DATEDIFF('day', s.last_login, CURRENT_DATE()),
             s.articles_read_30d,
             DATEDIFF('month', s.start_date, CURRENT_DATE()),
             s.avg_session_sec::FLOAT,
             s.paywall_bounces_30d
         )) c
    WHERE c.risk_segment = 'HIGH'
    ORDER BY s.ltv_estimated_eur DESC
    LIMIT 10
""").to_pandas()

print("=== Top 10 High-Value Subscribers at Risk ===")
print(results.to_string(index=False))
print()
print("These subscribers should receive a re-engagement campaign.")
print("Data never left Snowflake. No SageMaker. No inference endpoint.")

=== Top 10 High-Value Subscribers at Risk ===
   USER_ID SUBSCRIPTION_TYPE  LTV_ESTIMATED_EUR  CHURN_PROBABILITY RISK_SEGMENT
USR_019422          standard            1999.52             0.7228         HIGH
USR_046181           premium            1999.07             0.7941         HIGH
USR_045867          standard            1998.54             0.7300         HIGH
USR_043833           student            1998.10             0.7549         HIGH
USR_022410           student            1998.02             0.7338         HIGH
USR_041636          standard            1997.95             0.8336         HIGH
USR_057132          standard            1997.56             0.7452         HIGH
USR_010830          standard            1997.30             0.8933         HIGH
USR_051603          standard            1997.30             0.8423         HIGH
USR_022210           premium            1997.15             0.8196         HIGH

These subscribers should receive a re-engagement campaign.
Data never lef

## Summary

| Step | Where it runs | Compare to GCP |
|------|--------------|----------------|
| Feature engineering | Snowpark (inside Snowflake) | BigQuery ML or export to Vertex |
| Model training | Snowpark + scikit-learn | Vertex AI / SageMaker |
| Model storage | Snowflake Stage | GCS / S3 bucket |
| Inference | Python UDTF (SQL-callable) | Vertex endpoint / batch prediction |
| Governance | Same RBAC as all data | Separate IAM for ML |

**Key differentiator:** The model is a SQL function. Any analyst can call `PREDICT_CHURN(...)` without Python, without a separate ML platform, without moving data.